# 03. Denoising - DnCNN_Gray_Blind

| 항목 | 내용 |
|---|---|
| 대상 모델 | `DnCNN_Gray_Blind.dxnn` |
| 카테고리 | **Denoising** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 출력이 이름표가 아니라 영상일 때 후처리가 무엇을 하는지 설명할 수 있다.
2. `float32` 출력을 `uint8` 영상으로 되돌리는 계산을 실제 값으로 수행할 수 있다.
3. 출력 값 범위를 코드가 추측해 갈라내는 방식과 그 위험을 설명할 수 있다.
4. 노이즈가 줄었음을 원본과의 차이·표준편차로 측정해 제시할 수 있다.

## 1. Denoising (노이즈 제거)

**노이즈가 낀 영상을 받아 픽셀마다 원래 밝기를 추정해 깨끗한 영상을 만드는 작업**

답이 픽셀 하나하나의 값이므로 출력의 크기가 입력과 같다. 사진에 무엇이 찍혔는지는 판정하지 않고 밝기만
다시 계산한다.

**출력의 성격**

픽셀마다 값이 하나씩 나오는 영상이다. 이름표도 좌표도 없으므로 결과가 맞는지는 사람이 눈으로 보거나
원본과의 차이를 재서 판단한다. 크기는 그대로여서 해상도가 올라가지 않는다.

**활용 사례**

- 저조도·고감도 촬영 영상의 잡티 제거
- 의료·산업 영상의 판독 전 정리
- 압축으로 생긴 얼룩 완화

**포함되지 않는 작업**

- 해상도 확대 (Super Resolution)
- 어두운 영상의 밝기 복원 (Low-light Enhancement)
- 무엇이 찍혔는지 판정 (Image Classification)

## 2. 전체 파이프라인

```text
sample_denoising.jpg  512×512 BGR
  → 전처리         BGR→GRAY · 512×512 resize · 채널 축 추가 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → Public Output  80  [1, 1, 512, 512]  float32  (NCHW, 값은 0~1 부근)
  → 후처리         squeeze → 값 범위 판정 → clip 후 ×255 → uint8
  → Result         RestorationResult (깨끗해진 영상 한 장)
  → 시각화         원본과 결과를 좌우로 붙여 비교
```

![03. Denoising 전체 파이프라인](resources/03_Denoising_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input.1` |
| Shape / Dtype | `[1, 512, 512, 1]` / `uint8` |
| 채널 순서 | NHWC, 채널 1개 |
| Color / 값 범위 | Grayscale / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**다른 모델과의 차이점**

- **채널이 1개** — 컬러를 받지 않는다. 컬러 사진을 넣으면 흑백으로 바꿔 처리하므로 색이 사라진다
- **입력과 출력의 크기가 같음** — 512×512 로 들어가 512×512 로 나온다. 확대·축소가 목적이 아니다
- **`uint8` 입력** — 픽셀 값을 그대로 넣고, 0~1 로 나누는 일은 모델 안에서 처리된다

**`Blind` 의 뜻**

노이즈 제거 모델은 보통 노이즈 세기별로 따로 학습한다. 세기를 잘못 고르면 노이즈가 남거나 원래 무늬까지
뭉개진다. `Blind` 는 여러 세기를 섞어 학습해 **세기를 모른 채 쓸 수 있다**는 뜻이다. 실무에서는 노이즈가
얼마나 심한지 미리 알 수 없어 이쪽이 쓰기 편하고, 대신 세기 전용 모델보다 각 세기에서의 성능은 조금
떨어지는 것이 일반적이다.

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `80` | `[1, 1, 512, 512]` / `float32` | 깨끗해진 영상 한 장 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 이 모델은 Task 가 `npu_0` 하나뿐이라
NPU 가 계산한 값이 그대로 나온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 1 | 채널. 흑백이라 1개 |
| 2 | 512 | 높이 |
| 3 | 512 | 너비 |

**입력은 NHWC 인데 출력은 NCHW 다.** 채널 축이 뒤에서 앞으로 옮겨진 형태이며, `parse_model` 이 이 출력의
transpose 를 `CHANNEL_LAST_TO_FIRST` 로 보고한다. 채널이 1개라 축을 옮겨도 값의 순서가 바뀌지 않으므로
후처리는 크기 1인 축을 눌러 없애는 것으로 끝낸다.

`float32` 하나가 4바이트이므로 `512 × 512` 는 1,048,576 바이트다. 입력 256 KB 가 출력 1 MB 가 되는데,
값의 개수는 같고 `uint8` 이 `float32` 로 바뀌어 네 배가 된 것이다.

**원 논문은 노이즈만 예측해 입력에서 뺀다.** 사진 전체를 새로 그리는 것보다 덧씌워진 잡티만 알아내는 편이
배우기 쉬워 학습이 안정적이기 때문이며, 이 방식을 잔차(residual) 학습이라 부른다.

```text
원 논문:   노이즈 영상 - 예측한 노이즈 = 깨끗한 영상
이 배포본: 출력이 곧 깨끗한 영상        (뺄셈 단계가 없다)
```

**이 배포본은 깨끗한 영상을 곧바로 낸다.** 뺄셈이 모델 안에 들어가 컴파일됐거나 애초에 직접 내도록
만들어졌으며, 쓰는 쪽에서는 출력이 곧 결과다. 논문 구조와 배포 모델이 다를 수 있다는 예다.

#### 출력을 사람이 읽는 값으로 바꾸기

출력은 소수점 값이고 화면에 그릴 영상은 0~255 정수다. **그 소수점 값이 0~1 범위인지 0~255 범위인지는
모델 파일이 알려 주지 않으므로** 후처리가 실제 값의 최소·최대를 보고 갈라낸다.

| 실제 값 범위 | 하는 일 |
|---|---|
| 최대와 최소가 거의 같음 | 전부 0 으로 채움 (계산이 잘못된 경우) |
| 대략 0~1 | `clip(0, 1)` 후 `×255` |
| 대략 0~255 | `clip(0, 255)` 로 자르기만 |
| 그 밖 | 그 영상의 최소·최대로 0~255 에 펼침 |

**마지막 갈래가 위험하다.** 영상마다 기준이 달라져 어두운 영상은 밝게, 밝은 영상은 어둡게 조정된다. 보기에는
좋아지지만 절대 밝기가 보존되지 않아 여러 장을 비교하는 용도로는 쓸 수 없다.

이번 실행의 값은 최소 `0.0907`, 최대 `0.9156` 이라 두 번째 갈래로 들어간다. 0 미만과 1 초과가 하나도 없어
`clip` 이 실제로 잘라 내는 픽셀은 없고 `×255` 만 적용된다. 예를 들어 값 `0.4915` 는
`0.4915 × 255 = 125.3` 이므로 `uint8` 125 가 된다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 입력이 NHWC 1채널,
출력이 NCHW 이고 transpose 가 `CHANNEL_LAST_TO_FIRST` 인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "DnCNN_Gray_Blind.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정한다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_denoising.jpg"

print("input image:", IMAGE_PATH.name)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원 논문처럼 뺄셈을 하려면
  여기 담긴 0~1 정규화 입력이 필요하지만, 이 배포본의 후처리는 쓰지 않는다
- `RestorationResult` — 후처리가 만들어 시각화로 넘기는 결과. 영상 한 장을 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. grayscale 은 값 범위를 0~255 로 고정한다. 고정하지 않으면
  배열의 최소·최대를 양 끝으로 늘려, 대비가 낮아진 결과가 실제보다 선명한 그림으로 보인다

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0
    normalized_input: np.ndarray = field(default=None)


@dataclass
class RestorationResult:
    """복원된 영상 한 장. 원본 대응: common/processors/restoration_postprocessor.py"""

    output_image: np.ndarray = field(default=None)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. `parse_model` 출력과 같아야 한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 흑백 한 채널로 바꾸고 512×512 로 맞춘 뒤 채널 축을 붙여 `[512, 512, 1]` 로 만든다. 색을 버리는
단계가 여기이며, 컬러 사진을 넣어도 이 지점에서 색 정보가 사라진다.

확인할 것:

- 채널 — 3채널 BGR → 1채널 흑백
- 크기 — 512×512
- dtype — `uint8` 유지
- 축 구성 — `[H, W]` 에 채널 축을 붙여 `[H, W, 1]`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `dncnn_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `dncnn_preprocessor_process` — 흑백 변환·resize·채널 축 추가를 거쳐 입력 Tensor 와 Context 로 변환

In [ ]:
def dncnn_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def dncnn_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    gray = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (ctx.input_width, ctx.input_height))
    # 원 논문 방식으로 뺄셈을 하려면 0~1 정규화 입력이 필요하다. 이 배포본은 쓰지 않지만 원본과 같게 담아 둔다.
    ctx.normalized_input = resized.astype(np.float32) / 255.0

    return resized[:, :, np.newaxis], ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 흑백 영상의 밝기 분포를 함께 출력해 두면 결과와 비교할 기준이 된다.

In [ ]:
preprocessor_state = dncnn_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = dncnn_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"gray 분포     min={input_tensor.min()} max={input_tensor.max()} "
      f"mean={input_tensor.mean():.2f} std={input_tensor.std():.2f}")

notebook_show_bgr(input_tensor[:, :, 0], f"Preprocessed gray {INPUT_WIDTH}x{INPUT_HEIGHT}",
                  figsize=(6, 6))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- shape — `[1, 1, 512, 512]` 로 채널 축이 앞에 있다
- 값 범위 — 0~1 부근이라 그대로 화면에 그릴 수 없다
- 0 미만·1 초과 개수 — `clip` 이 실제로 잘라 낼 픽셀이 있는지

In [ ]:
raw = np.asarray(outputs[0])
notebook_describe_array("raw output", raw)

flat = raw.reshape(-1)
print(f"0 미만 = {int((flat < 0.0).sum())}개, 1 초과 = {int((flat > 1.0).sum())}개 / {flat.size}")
print(f"squeeze 뒤 shape = {np.squeeze(raw).shape}")

### 4.4 후처리

크기 1인 축을 눌러 `[512, 512]` 로 만든 뒤 값 범위를 판정해 `uint8` 영상으로 되돌린다. 순서는 squeeze →
범위 판정 → 변환이며, 판정 결과에 따라 곱하기·자르기·펼치기 중 하나만 적용된다.

**범위를 판정하는 것은 모델이 값의 의미를 알려 주지 않기 때문이다.** 그래서 실제 값의 최소·최대를 보고
어느 갈래인지 정한다. 이번 입력은 0~1 갈래로 들어가지만, 같은 코드가 다른 모델·다른 사진에서는 다른
갈래를 탈 수 있다는 점이 이 후처리의 성격이다.

이 배포본은 노이즈를 빼는 계산을 하지 않으므로 후처리가 Context 를 읽지 않는다. 원 논문 방식이라면
여기서 `정규화 입력 - 출력` 을 계산해야 한다.

#### 후처리 함수 정의

- `dncnn_postprocessor_create` — 이 모델은 담을 상수가 없어 빈 state 를 만듦
- `dncnn_to_uint8` — 값 범위를 판정해 `uint8` 로 변환
- `dncnn_postprocessor_process` — squeeze 와 변환을 거쳐 Result 생성

In [ ]:
def dncnn_postprocessor_create() -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {}


def dncnn_to_uint8(denoised: np.ndarray) -> np.ndarray:
    """값 범위를 판정해 0~255 uint8 영상으로 바꾼다. 원본 대응: DnCNNPostprocessor.process()"""
    dmin, dmax = float(denoised.min()), float(denoised.max())
    if dmax - dmin < 1e-6:
        return np.zeros_like(denoised, dtype=np.uint8)
    if dmin >= -0.1 and dmax <= 1.1:
        return (np.clip(denoised, 0.0, 1.0) * 255.0).astype(np.uint8)
    if dmin >= -1.0 and dmax <= 256.0:
        return np.clip(denoised, 0.0, 255.0).astype(np.uint8)
    return ((denoised - dmin) / (dmax - dmin) * 255.0).astype(np.uint8)


def dncnn_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    raw = np.squeeze(outputs[0])
    if raw.ndim == 3:
        # 컬러 모델이면 CHW 를 HWC 로 돌린다. 흑백인 이 모델은 이 분기를 타지 않는다.
        if not (raw.shape[2] <= 4 and raw.shape[0] > 4):
            raw = np.transpose(raw, (1, 2, 0))
    return [RestorationResult(output_image=dncnn_to_uint8(raw))]

#### 후처리 실행

확인할 것:

- dtype·shape — `uint8` `[512, 512]`
- 밝기 평균 — 입력과 크게 달라지지 않아야 한다
- 표준편차 — 노이즈가 빠지면 값이 덜 흩어져 작아진다
- 원본과의 차이 — 얼마나 많은 픽셀이 얼마나 바뀌었는지

In [ ]:
postprocessor_state = dncnn_postprocessor_create()
results = dncnn_postprocessor_process(postprocessor_state, outputs, ctx)
denoised = results[0].output_image

print(f"denoised      shape={denoised.shape} dtype={denoised.dtype} "
      f"min={denoised.min()} max={denoised.max()}")
print(f"밝기 평균     입력={input_tensor[:, :, 0].mean():.2f} → 결과={denoised.mean():.2f}")
print(f"표준편차      입력={input_tensor[:, :, 0].std():.2f} → 결과={denoised.std():.2f}")

diff = np.abs(input_tensor[:, :, 0].astype(np.int16) - denoised.astype(np.int16))
print(f"|입력-결과|   max={diff.max()} mean={diff.mean():.2f} "
      f"5 초과 픽셀={(diff > 5).mean() * 100:.1f}%")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 원본과 결과를 좌우로 붙인다. 흑백 결과는 3채널로 늘려 컬러 원본과 나란히 놓을 수 있게
하고, 결과를 원본 크기로 되돌린 뒤 붙인다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글
자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def dncnn_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본과 좌우로 붙여 그린다. 원본 대응: Runner.visualize()"""
    denoised = results[0].output_image
    if denoised.ndim == 2:
        denoised = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)

    height, width = image.shape[:2]
    canvas = np.hstack([image, cv2.resize(denoised, (width, height))])

    cv2.putText(canvas, "Input", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.putText(canvas, "Denoised", (width + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2)
    return canvas

#### 결과 시각화

좌우 비교를 inline 으로 표시한다. 왼쪽은 컬러 원본이고 오른쪽은 흑백 결과를 3채널로 늘린 것이라 색이
사라진 것도 함께 보인다.

In [ ]:
output_image = dncnn_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Input vs Denoised", figsize=(12, 6))

#### 같은 자리를 확대해 비교

노이즈 제거는 전체를 축소해 보면 차이가 잘 드러나지 않는다. 같은 좌표의 128×128 영역을 네 배로 확대해
나란히 놓으면 잡티가 사라진 정도와 함께 무늬가 뭉개진 정도도 보인다.

In [ ]:
CROP_Y, CROP_X, CROP = 192, 192, 128
noisy_crop = input_tensor[CROP_Y:CROP_Y + CROP, CROP_X:CROP_X + CROP, 0]
clean_crop = denoised[CROP_Y:CROP_Y + CROP, CROP_X:CROP_X + CROP]

zoom = cv2.resize(np.hstack([noisy_crop, clean_crop]), None, fx=4, fy=4,
                  interpolation=cv2.INTER_NEAREST)
print(f"crop 표준편차  입력={noisy_crop.std():.2f} → 결과={clean_crop.std():.2f}")
notebook_show_bgr(zoom, "Zoom x4 - noisy (left) vs denoised (right)", figsize=(12, 6))